# Baselines MLP para AskMind

Este notebook reemplaza la logica de la propuesta anterior basada en `policy_dataset_v2.xlsm` por un pipeline alineado con AskMind.

## Objetivo general

Construir baselines MLP para una politica conversacional binaria que decida si el sistema debe:

- `ask`: pedir una aclaracion adicional
- `respond`: responder con el contexto disponible

## Hoja de ruta del notebook

1. Exploracion de los datos originales (`train.jsonl` y `test.jsonl`).
2. Descripcion de la tarea de decision y de la metodologia de transformacion del dataset.
3. Construccion del dataset tabular que realmente consume el MLP.
4. Representacion del estado mediante embeddings.
5. Diseño de los baselines `mlp_policy_gradient` y `mlp_q_learning`.
6. Resultados en validacion e inferencia exploratoria sobre el test oficial.

## Notas importantes

- Este notebook es **autocontenido**: la seccion 0 descarga el dataset AskMind desde Hugging Face si `askmind_data/` no esta presente y reproduce el preprocesamiento original.
- El split de validacion se construye desde `train.jsonl`, porque no existe dev oficial.
- El `test.jsonl` oficial no trae trayectoria conversacional anotada, asi que aqui se usa solo para inferencia exploratoria.
- Si luego recibes embeddings externos, este notebook permite reemplazar el fallback TF-IDF + SVD por un archivo `.npz`.

In [18]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy",
        "pandas",
        "scikit-learn",
        "torch"
    ])

In [2]:
from __future__ import annotations

import hashlib
import html
import json
import math
import re
import textwrap
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_SEED = 42
ACTION_TO_INDEX = {"ask": 0, "respond": 1}
INDEX_TO_ACTION = {index: action for action, index in ACTION_TO_INDEX.items()}

def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@dataclass
class BaselineConfig:
    data_dir: Path = Path("askmind_data")
    validation_fraction: float = 0.20
    random_seed: int = RANDOM_SEED
    tfidf_max_features: int = 4096
    embedding_dim: int = 256
    batch_size: int = 128
    hidden_dims: tuple[int, ...] = (256, 128)
    dropout: float = 0.10
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    policy_entropy_coef: float = 0.01
    q_discount_factor: float = 0.90
    epochs: int = 5
    patience: int = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    precomputed_embeddings: Path | None = None

config = BaselineConfig(
    data_dir=Path("askmind_data"),
    epochs=5,
    embedding_dim=256,
    precomputed_embeddings=None,
 )

set_seed(config.random_seed)
config

BaselineConfig(data_dir=PosixPath('askmind_data'), validation_fraction=0.2, random_seed=42, tfidf_max_features=4096, embedding_dim=256, batch_size=128, hidden_dims=(256, 128), dropout=0.1, learning_rate=0.001, weight_decay=0.0001, policy_entropy_coef=0.01, q_discount_factor=0.9, epochs=5, patience=4, device='cpu', precomputed_embeddings=None)

## 0. Descarga automatica del dataset

Este notebook es **autocontenido**: la siguiente celda descarga el dataset AskMind
desde Hugging Face si no esta presente y reproduce el mismo preprocesamiento del
proyecto (se filtran filas con caracteres Han/CJK). Si los archivos ya existen en
`askmind_data/`, la descarga se omite. Para forzar la redescarga usa
`ensure_askmind_dataset(config.data_dir, force=True)`.


In [3]:
# === 0. Descarga automatica del dataset AskMind (notebook autocontenido) ===
# Si askmind_data/{train,test}.jsonl no existen, se descargan los archivos crudos
# de AskBench desde Hugging Face y se aplica el preprocesamiento del proyecto
# (se eliminan filas con caracteres Han/CJK). Solo usa la libreria estandar.
import urllib.request

ASKMIND_SOURCES = {
    "train.jsonl": {
        "url": "https://huggingface.co/datasets/jialeuuz/askbench_train/resolve/main/mind.jsonl",
        "source": "jialeuuz/askbench_train/mind.jsonl",
        "expected": 5830,
    },
    "test.jsonl": {
        "url": "https://huggingface.co/datasets/jialeuuz/askbench_bench/resolve/main/ask_bench_data/ask_mind.jsonl",
        "source": "jialeuuz/askbench_bench/ask_bench_data/ask_mind.jsonl",
        "expected": 399,
    },
}

# Caracteres Han/CJK: Ext-A, Unified Ideographs y Compatibility Ideographs.
_HAN_PATTERN = re.compile(r"[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff]")


def _download_text(url: str, timeout: int = 180) -> str:
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read().decode("utf-8")


def _clean_jsonl(raw_text: str) -> list[str]:
    """Replica el preprocesamiento del proyecto: descarta filas con Han/CJK,
    lineas vacias y registros que no parseen como JSON valido."""
    cleaned = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line or _HAN_PATTERN.search(line):
            continue
        try:
            json.loads(line)
        except json.JSONDecodeError:
            continue
        cleaned.append(line)
    return cleaned


def ensure_askmind_dataset(data_dir: Path, force: bool = False) -> pd.DataFrame:
    """Garantiza que existan train.jsonl y test.jsonl en `data_dir`, descargando
    y reconstruyendo desde Hugging Face solo si faltan (o si force=True)."""
    data_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for filename, meta in ASKMIND_SOURCES.items():
        target = data_dir / filename
        split = filename.split(".")[0]
        if target.exists() and target.stat().st_size > 0 and not force:
            count = sum(1 for line in target.open(encoding="utf-8") if line.strip())
            print(f"[skip]  {filename}: ya presente ({count} ejemplos)")
        else:
            print(f"[descargando] {meta['source']} -> {filename}")
            cleaned = _clean_jsonl(_download_text(meta["url"]))
            target.write_text("\n".join(cleaned) + "\n", encoding="utf-8")
            count = len(cleaned)
            warn = "" if count == meta["expected"] else f"  [!] esperado {meta['expected']}"
            print(f"[listo] {filename}: {count} ejemplos{warn}")
        rows.append({"file": filename, "split": split, "examples": count, "source": meta["source"]})

    manifest = pd.DataFrame(rows)
    manifest["preprocessing"] = "Removed rows containing Chinese/Han characters anywhere in JSON row"
    manifest.to_csv(data_dir / "MANIFEST.csv", index=False)
    return manifest


dataset_manifest = ensure_askmind_dataset(config.data_dir)
display(dataset_manifest)


[descargando] jialeuuz/askbench_train/mind.jsonl -> train.jsonl
[listo] train.jsonl: 5830 ejemplos
[descargando] jialeuuz/askbench_bench/ask_bench_data/ask_mind.jsonl -> test.jsonl
[listo] test.jsonl: 399 ejemplos


,file,split,examples,source,preprocessing
0,train.jsonl,train,5830,jialeuuz/askbench_train/mind.jsonl,Removed rows containing Chinese/Han characters...
1,test.jsonl,test,399,jialeuuz/askbench_bench/ask_bench_data/ask_min...,Removed rows containing Chinese/Han characters...


## 1. Exploracion previa de train y test

Antes de convertir AskMind en decisiones `ask/respond`, conviene inspeccionar la forma original de los archivos.

- `train.jsonl` mezcla preguntas completas y conversaciones degradadas con `conversation_history`.
- `test.jsonl` no trae trayectoria conversacional, asi que sirve para exploracion e inferencia, no para medir la politica con labels comparables.
- Esta revision ayuda a justificar por que la validacion se construye a partir de `train`.


In [4]:
def compact_text(text: str, limit: int = 110) -> str:
    clean = re.sub(r"\s+", " ", text or "").strip()
    return clean if len(clean) <= limit else clean[: limit - 3] + "..."


def summarize_raw_split(rows: list[dict], split_name: str) -> dict:
    history_lengths = pd.Series([len(row.get("conversation_history") or []) for row in rows], dtype="int64")
    required_lengths = pd.Series([len(row.get("required_points") or []) for row in rows], dtype="int64")
    degraded_lengths = pd.Series([len(compact_text(row.get("degraded_question", "")).split()) for row in rows], dtype="int64")
    return {
        "split": split_name,
        "rows": len(rows),
        "rows_with_history": int((history_lengths > 0).sum()),
        "rows_without_history": int((history_lengths == 0).sum()),
        "avg_history_turns": float(history_lengths.mean()),
        "max_history_turns": int(history_lengths.max()),
        "avg_required_points": float(required_lengths.mean()),
        "max_required_points": int(required_lengths.max()),
        "avg_degraded_question_tokens": float(degraded_lengths.mean()),
    }


def build_field_presence(rows: list[dict], split_name: str, fields: list[str]) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "field": fields,
            split_name: [
                round(100.0 * sum(bool(row.get(field)) for row in rows) / len(rows), 2)
                for field in fields
            ],
        }
    )


def preview_rows(rows: list[dict], split_name: str, limit: int = 3) -> pd.DataFrame:
    preview = []
    for row in rows[:limit]:
        history = row.get("conversation_history") or []
        preview.append({
            "split": split_name,
            "id": str(row.get("id", ""))[:12],
            "has_history": bool(history),
            "history_turns": len(history),
            "required_points_total": len(row.get("required_points") or []),
            "degraded_question": compact_text(row.get("degraded_question", "")),
        })
    return pd.DataFrame(preview)


train_raw_rows = [
    json.loads(line)
    for line in (config.data_dir / "train.jsonl").open(encoding="utf-8")
    if line.strip()
]
test_raw_rows = [
    json.loads(line)
    for line in (config.data_dir / "test.jsonl").open(encoding="utf-8")
    if line.strip()
]

raw_split_summary = pd.DataFrame([
    summarize_raw_split(train_raw_rows, "train"),
    summarize_raw_split(test_raw_rows, "test"),
]).round(2)

presence_fields = [
    "ori_question",
    "degraded_question",
    "conversation_history",
    "required_points",
    "expected_answer",
    "pass_rate",
    "category",
]
field_presence = build_field_presence(train_raw_rows, "train", presence_fields).merge(
    build_field_presence(test_raw_rows, "test", presence_fields),
    on="field",
    how="outer",
)

train_history_role_counts = pd.Series(
    [message.get("role", "unknown") for row in train_raw_rows for message in (row.get("conversation_history") or [])]
).value_counts().rename_axis("role").reset_index(name="count")

sample_preview = pd.concat(
    [preview_rows(train_raw_rows, "train"), preview_rows(test_raw_rows, "test")],
    ignore_index=True,
 )

print("Resumen estructural de los archivos originales")
display(raw_split_summary)

print("Porcentaje de filas donde aparece cada campo")
display(field_presence)

print("Distribucion de roles dentro de las conversaciones de train")
display(train_history_role_counts)

print("Muestra rapida de filas originales")
display(sample_preview)

Resumen estructural de los archivos originales


,split,rows,rows_with_history,rows_without_history,avg_history_turns,max_history_turns,avg_required_points,max_required_points,avg_degraded_question_tokens
0,train,5830,2909,2921,3.2,10,1.57,7,9.40
1,test,399,0,399,0.0,0,3.85,10,16.76


Porcentaje de filas donde aparece cada campo


,field,train,test
0,category,0.0,25.06
1,conversation_history,49.9,0.00
2,degraded_question,49.9,100.00
3,expected_answer,100.0,100.00
4,ori_question,100.0,100.00
5,pass_rate,94.1,0.00
6,required_points,49.9,100.00


Distribucion de roles dentro de las conversaciones de train


,role,count
0,user,9325
1,assistant,9325


Muestra rapida de filas originales


,split,id,has_history,history_turns,required_points_total,degraded_question
0,train,779cb8a9ce5f,True,4,3,Bernardo randomly picks a few distinct numbers...
1,train,,False,0,0,
2,train,3a1ec0f3e78a,True,4,3,Find the smallest positive integer $n$ for whi...
3,test,e9b1f5f379ce,False,0,6,Please answer the following multiple-choice qu...
4,test,7fa67fde15ce,False,0,3,Particles are collided at the center of a sphe...
5,test,92ea9b547826,False,0,3,not not not not not not not True is


## 1.1 Estructura de los datasets originales

Antes de transformar AskMind en ejemplos por turno, conviene dejar explicito que los archivos originales no tienen una sola forma tabular.

### Vista general por archivo

| Dataset original | Unidad observada | Estructura dominante | Campos caracteristicos | Implicacion para el notebook |
| --- | --- | --- | --- | --- |
| `train.jsonl` | una fila JSON por ejemplo | mezcla de dos formatos: preguntas completas y trayectorias multi-turn degradadas | `ori_question`, `expected_answer` y, en parte del archivo, `degraded_question`, `degraded_info`, `required_points`, `conversation_history`, `pass_rate` | no se puede usar directamente como clasificacion `ask/respond`; primero hay que convertirlo en decisiones por turno |
| `test.jsonl` | una fila JSON por ejemplo | prompts degradados sin trayectoria conversacional anotada | `id`, `ori_question`, `degraded_question`, `degraded_info`, `required_points`, `expected_answer`; en algunos casos tambien `category`, `err_info`, `solution`, `source_task` | sirve para exploracion e inferencia, pero no para construir supervision por turnos comparable con train |

### Significado y estructura de los campos

| Campo | Que significa | Estructura original | Donde aparece | Como se usa en este notebook |
| --- | --- | --- | --- | --- |
| `id` | identificador del ejemplo original | string hash o identificador unico | principalmente en `test`, y en parte de `train` | trazabilidad y construccion de `example_id` derivados |
| `ori_question` | pregunta completa sin degradacion | string | `train` y `test` | agrupacion para split sin fuga y referencia semantica del problema |
| `degraded_question` | version incompleta o ambigua de la pregunta | string | filas degradadas de `train` y todo `test` | base del estado que decide si preguntar o responder |
| `degraded_info` | descripcion textual de la informacion removida o degradada | string o lista serializada segun el registro | `train` degradado y `test` | contexto descriptivo del dataset; aqui se documenta pero no entra al feature principal |
| `conversation_history` | trayectoria observada de la conversacion degradada | lista de mensajes; cada mensaje es un objeto con `role` y `content` | solo en parte de `train` | se recorre para generar ejemplos supervisados por turno |
| `required_points` | piezas de informacion que deberian aclararse para responder bien | lista de strings | `train` degradado y `test` | heuristica para cobertura y shaping de rewards |
| `expected_answer` | respuesta final de referencia del benchmark | string | `train` y `test` | referencia cualitativa; no se usa como input del estado |
| `pass_rate` | senal de dificultad o tasa de exito del benchmark original | numerico | solo en parte de `train` | queda disponible como metadata, pero no participa en el baseline actual |
| `category` | categoria tematica del item | string | algunos ejemplos de `test` | exploracion descriptiva del test oficial |
| `err_info` | informacion adicional sobre el tipo de error o perturbacion | string o null | algunos ejemplos de `test` | exploracion descriptiva; no entra al entrenamiento |
| `solution` | solucion o desarrollo de referencia del problema | string | algunos ejemplos de `test` | inspeccion cualitativa del benchmark |
| `source_task` | procedencia del item dentro del benchmark original | string | algunos ejemplos de `test` | analisis exploratorio del test |

### Forma de `conversation_history`

`conversation_history` no es texto plano. Viene como una lista ordenada de mensajes con esta forma conceptual:

```json
[
  {"role": "user", "content": "pregunta degradada o aclaracion"},
  {"role": "assistant", "content": "pregunta de aclaracion o respuesta final"}
]
```

Esa estructura es la que luego el preprocesamiento serializa como `CONVERSATION_SO_FAR` para construir el estado textual del modelo.

## 2. Tarea de decision del MLP

El MLP no genera texto ni resuelve directamente la pregunta del usuario. Su trabajo es tomar una decision de control conversacional antes de que exista una respuesta final.

La decision es binaria:

- `ask`: el sistema todavia necesita una aclaracion porque la pregunta degradada sigue dejando informacion importante sin resolver.
- `respond`: el sistema considera que ya hay suficiente contexto para responder.

En otras palabras, el modelo aprende una politica simple de timing conversacional: decidir si conviene seguir preguntando o si ya es momento de responder.

### Como se representa esa decision

La salida del modelo siempre compara dos acciones posibles sobre el mismo estado:

- estado actual -> `ask`
- estado actual -> `respond`

El estado se construye con dos piezas:

- `degraded_question`
- `conversation_history_so_far`

Eso hace que el MLP aprenda sobre el contexto acumulado, no solo sobre la pregunta inicial aislada.

## 3. Procesamiento y metodologia de transformacion

El preprocesamiento convierte AskMind en ejemplos supervisados por turno. Esa es la parte clave del notebook, porque el dataset original no viene como una tabla lista para clasificacion `ask/respond`.

### Paso 1: separar los dos tipos de filas

En `train.jsonl` aparecen dos formatos distintos:

- filas con `conversation_history`, que contienen una trayectoria degradada con aclaraciones y respuesta final
- filas con `ori_question` sin trayectoria, que representan preguntas completas donde responder es la accion natural

### Paso 2: convertir conversaciones en ejemplos por turno

Para cada conversacion se recorre el historial y se construye un ejemplo cada vez que habla el asistente:

- si no es el ultimo turno del asistente, ese ejemplo se etiqueta como `ask`
- si es el ultimo turno del asistente, ese ejemplo se etiqueta como `respond`

Asi el problema queda reformulado como una secuencia de decisiones locales: en este punto de la conversacion, que debio hacer el sistema.

### Paso 3: construir el estado textual

Cada ejemplo se serializa como un bloque con:

- `QUESTION`: la version degradada de la pregunta
- `CONVERSATION_SO_FAR`: solo el historial disponible hasta ese turno

Eso evita fuga de informacion, porque el modelo no ve mensajes futuros al momento de decidir.

### Paso 4: aproximar rewards con `required_points`

El notebook no recibe rewards densos ya hechos. Por eso construye una senal simple:

- para `ask`, se recompensa mas cuando la siguiente respuesta del usuario ayuda a cubrir `required_points`
- para `respond`, se penaliza si todavia queda informacion importante sin cubrir
- en el ultimo turno, `respond` recibe reward alto porque ya corresponde contestar

### Como se disena el reward en cada turno-accion

La idea no es premiar texto bonito, sino premiar decisiones de control conversacional consistentes con el estado parcial del dialogo.

Para cada estado del asistente se calculan dos rewards posibles, uno por accion:

- `ask_reward`: cuanto valdria seguir preguntando en ese punto
- `respond_reward`: cuanto valdria responder ya en ese punto

#### Caso 1: el turno gold es `ask`

Eso significa que todavia no era momento de responder. Entonces el notebook mira la siguiente intervencion del usuario y mide si esa aclaracion realmente ayudo a cubrir mas `required_points`.

1. Se calcula `covered_before`: cuantos puntos ya estaban cubiertos antes de preguntar.
2. Se busca la siguiente respuesta del usuario.
3. Se calcula `covered_after`: cuantos puntos quedan cubiertos despues de incorporar esa respuesta.
4. Se define `coverage_gain = max(0, covered_after - covered_before)`.

Con eso se asigna:

- `ask_reward = 0.5 + 0.5 * (coverage_gain / total_points)`
- `respond_reward = - unresolved_fraction`

Interpretacion:

- `ask_reward` nunca parte de 0, porque si el turno gold era `ask`, ya existe evidencia de que preguntar era razonable.
- si la pregunta logra destapar informacion util, el reward sube por encima de 0.5.
- `respond_reward` se vuelve mas negativo cuando todavia falta cubrir una fraccion grande de los `required_points`.

#### Caso 2: el turno gold es `respond`

Eso significa que el asistente ya tiene suficiente contexto para cerrar la interaccion. En ese caso el reward favorece con claridad la accion terminal:

- `respond_reward = 1.0`
- `ask_reward = -0.5` si existen `required_points`
- `ask_reward = -0.25` si no existen `required_points`

Interpretacion:

- `respond_reward = 1.0` fija una senal fuerte de cierre correcto.
- `ask_reward` es negativo porque seguir preguntando en ese punto introduce costo conversacional innecesario.
- la penalizacion es un poco menor cuando no hay `required_points`, porque ahi la estructura de supervision es mas debil y proviene de preguntas originales sin trayectoria degradada.

#### Por que este diseno es razonable

Este shaping busca aproximar tres preferencias del sistema:

- preguntar solo cuando la aclaracion agrega informacion util
- evitar respuestas prematuras cuando aun faltan datos importantes
- evitar preguntas redundantes cuando ya corresponde responder

No es un reward perfecto del mundo real, pero si una heuristica consistente con el objetivo de politica `ask/respond` y con las senales que AskMind realmente trae anotadas.

### Paso 5: hacer un split sin fuga

El split de validacion se hace agrupando por `ori_question`. Eso evita que variantes de la misma pregunta queden repartidas entre train y validation.

In [5]:
def normalize_whitespace(text: str) -> str:
    """Limpia espacios en blanco repetidos y recorta extremos.

    Por que se usa:
    - Estandariza texto de entrada para evitar diferencias por formato.
    - Mejora consistencia en serializacion, tokenizacion y comparacion.
    """
    return re.sub(r"\s+", " ", text or "").strip()


def normalize_text(text: str) -> str:
    """Normaliza texto para matching simple: minusculas y sin puntuacion.

    Por que se usa:
    - Reduce variaciones superficiales (mayusculas/simbolos).
    - Facilita chequeo de cobertura entre `required_points` y evidencia.
    """
    text = normalize_whitespace(text).lower()
    return re.sub(r"[^a-z0-9\s]", " ", text)


def tokenize(text: str) -> list[str]:
    """Convierte texto normalizado en tokens y filtra tokens muy cortos.

    Por que se usa:
    - Permite medir solapamiento lexico entre punto y evidencia.
    - Filtrar tokens de longitud <= 2 reduce ruido (ej. 'de', 'la', 'to').
    """
    return [token for token in normalize_text(text).split() if len(token) > 2]


def format_history(messages: Iterable[dict]) -> str:
    """Formatea el historial conversacional como bloque de texto por lineas.

    Cada linea queda como: `ROLE: contenido`. Se usa join con saltos de linea para preservar la estructura temporal del dialogo.

    Por que se usa:
    - Construye una representacion textual estable del contexto previo.
    - Se integra en `state_text` para embedding y entrenamiento del MLP.
    """
    parts = []
    for message in messages:
        role = (message.get("role") or "unknown").upper()
        content = normalize_whitespace(message.get("content", ""))
        parts.append(f"{role}: {content}")
    return "\n".join(parts)


def make_example_id(parts: list[str]) -> str:
    """Genera un ID deterministico (SHA1) a partir de partes de contexto.

    Por que se usa:
    - Identifica ejemplos de forma reproducible entre ejecuciones.
    - Sirve como clave para embeddings precomputados y trazabilidad.
    """
    joined = "||".join(parts)
    return hashlib.sha1(joined.encode("utf-8")).hexdigest()


def point_is_covered(point: str, evidence: str) -> bool:
    """Estima si un `required_point` esta cubierto por la evidencia textual.

    Regla:
    - Match directo de substring normalizado, o
    - Solapamiento por tokens con umbral:
      * 1 token si el punto tiene <= 2 tokens
      * 2 tokens en caso contrario

    Por que se usa:
    - Provee una senal heuristica de cobertura sin anotacion extra.
    - Alimenta el shaping de reward para acciones `ask/respond`.
    """
    evidence_tokens = set(tokenize(evidence))
    point_tokens = tokenize(point)
    if not point_tokens:
        return False
    if normalize_text(point) in normalize_text(evidence):
        return True
    overlap = sum(token in evidence_tokens for token in point_tokens)
    threshold = 1 if len(point_tokens) <= 2 else 2
    return overlap >= threshold


def count_covered_points(required_points: list[str], evidence: str) -> int:
    """Cuenta cuantos `required_points` aparecen cubiertos en la evidencia.

    Por que se usa:
    - Resume cobertura en un valor escalar.
    - Se utiliza para calcular `covered_before`, `coverage_gain` y rewards.
    """
    return sum(point_is_covered(point, evidence) for point in required_points)


def serialize_state_text(question_text: str, history_prefix: list[dict]) -> str:
    """Serializa estado conversacional en un formato textual fijo.

    Estructura:
    - `QUESTION:`
    - `CONVERSATION_SO_FAR:`

    Por que se usa:
    - Define la entrada del encoder textual (TF-IDF + SVD o embeddings externos).
    - Evita fuga de informacion al usar solo historial disponible hasta el turno.
    """
    question_block = normalize_whitespace(question_text)
    history_block = format_history(history_prefix) if history_prefix else "NO_HISTORY"
    return f"QUESTION:\n{question_block}\n\nCONVERSATION_SO_FAR:\n{history_block}"

In [6]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def build_rich_turn_examples(row: dict, split_name: str) -> list[dict]:
    history = row.get("conversation_history") or []
    degraded_question = row.get("degraded_question") or row.get("ori_question") or ""
    required_points = row.get("required_points") or []
    row_key = str(row.get("id") or row.get("ori_question") or degraded_question)
    trajectory_id = make_example_id([split_name, row_key, "trajectory"])
    examples = []
    running_history: list[dict] = []
    observed_user_evidence = []

    for index, message in enumerate(history):
        role = message.get("role")
        if role == "user":
            observed_user_evidence.append(message.get("content", ""))
            running_history.append(message)
            continue

        if role != "assistant":
            running_history.append(message)
            continue

        assistant_positions = [
            idx for idx, item in enumerate(history) if item.get("role") == "assistant"
        ]
        assistant_rank = assistant_positions.index(index)
        is_final_assistant = assistant_rank == len(assistant_positions) - 1
        action = "respond" if is_final_assistant else "ask"
        evidence_before = " ".join(observed_user_evidence)
        covered_before = count_covered_points(required_points, evidence_before)
        total_points = max(len(required_points), 1)
        unresolved_fraction = (len(required_points) - covered_before) / total_points if required_points else 0.0

        if action == "ask":
            next_user_content = ""
            for next_message in history[index + 1 :]:
                if next_message.get("role") == "user":
                    next_user_content = next_message.get("content", "")
                    break
            covered_after = count_covered_points(required_points, f"{evidence_before} {next_user_content}")
            coverage_gain = max(0, covered_after - covered_before)
            ask_reward = 0.5 + 0.5 * (coverage_gain / total_points)
            respond_reward = -float(unresolved_fraction)
        else:
            ask_reward = -0.5 if required_points else -0.25
            respond_reward = 1.0

        state_text = serialize_state_text(degraded_question, running_history)
        example_id = make_example_id([
            split_name,
            row_key,
            str(assistant_rank),
            action,
        ])
        examples.append({
            "example_id": example_id,
            "source_split": split_name,
            "source_kind": "degraded_conversation",
            "trajectory_id": trajectory_id,
            "step_index": assistant_rank,
            "is_terminal": is_final_assistant,
            "ori_question": row.get("ori_question", ""),
            "state_text": state_text,
            "action": action,
            "label": ACTION_TO_INDEX[action],
            "ask_reward": float(ask_reward),
            "respond_reward": float(respond_reward),
            "turn_index": assistant_rank,
            "required_points_total": len(required_points),
            "covered_before": covered_before,
            "gold_response_preview": normalize_whitespace(message.get("content", ""))[:200],
        })
        running_history.append(message)

    return examples

def build_original_question_examples(row: dict, split_name: str) -> list[dict]:
    if row.get("conversation_history"):
        return []
    if row.get("degraded_question"):
        return []

    ori_question = row.get("ori_question") or ""
    if not ori_question:
        return []

    state_text = serialize_state_text(ori_question, [])
    example_id = make_example_id([split_name, ori_question, "original", "respond"])
    trajectory_id = make_example_id([split_name, ori_question, "original_trajectory"])
    return [{
        "example_id": example_id,
        "source_split": split_name,
        "source_kind": "original_question",
        "trajectory_id": trajectory_id,
        "step_index": 0,
        "is_terminal": True,
        "ori_question": ori_question,
        "state_text": state_text,
        "action": "respond",
        "label": ACTION_TO_INDEX["respond"],
        "ask_reward": -0.25,
        "respond_reward": 1.0,
        "turn_index": 0,
        "required_points_total": 0,
        "covered_before": 0,
        "gold_response_preview": "",
    }]

def build_training_examples(rows: list[dict], split_name: str) -> pd.DataFrame:
    records = []
    for row in rows:
        if row.get("conversation_history"):
            records.extend(build_rich_turn_examples(row, split_name))
        else:
            records.extend(build_original_question_examples(row, split_name))
    return pd.DataFrame.from_records(records)

def build_official_test_examples(rows: list[dict]) -> pd.DataFrame:
    records = []
    for row in rows:
        degraded_question = row.get("degraded_question") or row.get("ori_question") or ""
        state_text = serialize_state_text(degraded_question, [])
        example_id = make_example_id(["official_test", row.get("id") or degraded_question])
        trajectory_id = make_example_id(["official_test", row.get("id") or degraded_question, "trajectory"])
        records.append({
            "example_id": example_id,
            "source_split": "official_test",
            "source_kind": "official_test_prompt",
            "ori_question": row.get("ori_question", ""),
            "state_text": state_text,
            "required_points_total": len(row.get("required_points") or []),
        })
    return pd.DataFrame.from_records(records)

def grouped_train_validation_split(examples: pd.DataFrame, config: BaselineConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    group_series = examples["ori_question"].fillna("")
    unique_groups = sorted(group_series.unique())
    rng = np.random.default_rng(config.random_seed)
    permutation = rng.permutation(len(unique_groups))
    validation_count = max(1, int(math.ceil(len(unique_groups) * config.validation_fraction)))
    validation_groups = set(unique_groups[index] for index in permutation[:validation_count])

    validation_mask = group_series.isin(validation_groups)
    train_df = examples.loc[~validation_mask].reset_index(drop=True)
    validation_df = examples.loc[validation_mask].reset_index(drop=True)
    return train_df, validation_df

def print_split_summary(train_df: pd.DataFrame, validation_df: pd.DataFrame) -> pd.DataFrame:
    summary = pd.DataFrame({
        "split": ["train", "validation"],
        "examples": [len(train_df), len(validation_df)],
        "ask_examples": [
            int((train_df["action"] == "ask").sum()),
            int((validation_df["action"] == "ask").sum()),
        ],
        "respond_examples": [
            int((train_df["action"] == "respond").sum()),
            int((validation_df["action"] == "respond").sum()),
        ],
    })
    print("Resumen de splits por turno")
    print(summary.to_string(index=False))
    return summary

### 3.1 Construccion del dataset tabular del MLP

Las siguientes celdas ejecutan la transformacion descrita arriba: cargan `train.jsonl`, crean una fila por decision del asistente, separan train/validacion por `ori_question` y muestran un registro original transformado paso a paso.

In [7]:
train_rows = load_jsonl(config.data_dir / "train.jsonl")
official_test_rows = load_jsonl(config.data_dir / "test.jsonl")

all_examples = build_training_examples(train_rows, split_name="train")
if all_examples.empty:
    raise RuntimeError("No se pudieron construir ejemplos de entrenamiento a partir de AskMind.")

train_df, validation_df = grouped_train_validation_split(all_examples, config)
split_summary = print_split_summary(train_df, validation_df)

display(split_summary)
display(
    train_df[[
        "source_kind",
        "action",
        "turn_index",
        "required_points_total",
        "covered_before",
        "ask_reward",
        "respond_reward",
    ]].head(5)
)

Resumen de splits por turno
     split  examples  ask_examples  respond_examples
     train      9809          5148              4661
validation      2437          1268              1169


,split,examples,ask_examples,respond_examples
0,train,9809,5148,4661
1,validation,2437,1268,1169


,source_kind,action,turn_index,required_points_total,covered_before,ask_reward,respond_reward
0,degraded_conversation,ask,0,3,1,0.666667,-0.666667
1,degraded_conversation,respond,1,3,2,-0.500000,1.000000
2,original_question,respond,0,0,0,-0.250000,1.000000
3,degraded_conversation,ask,0,3,2,0.666667,-0.333333
4,degraded_conversation,respond,1,3,3,-0.500000,1.000000


In [8]:
PRINT_WIDTH = 90


def wrap_text(
    text: object,
    width: int = PRINT_WIDTH,
    initial_indent: str = "",
    subsequent_indent: str | None = None,
    normalize: bool = True,
 ) -> str:
    if subsequent_indent is None:
        subsequent_indent = initial_indent
    text_value = str(text)
    if normalize:
        text_value = normalize_whitespace(text_value)
    if not text_value:
        return initial_indent.rstrip()
    return textwrap.fill(
        text_value,
        width=width,
        initial_indent=initial_indent,
        subsequent_indent=subsequent_indent,
        break_long_words=True,
        break_on_hyphens=False,
    )


def print_wrapped(
    text: object = "",
    initial_indent: str = "",
    subsequent_indent: str | None = None,
    normalize: bool = True,
 ) -> None:
    print(
        wrap_text(
            text,
            initial_indent=initial_indent,
            subsequent_indent=subsequent_indent,
            normalize=normalize,
        )
    )


def print_rule(char: str = "=") -> None:
    print(char * PRINT_WIDTH)


def print_section(title: str, char: str = "=") -> None:
    print_rule(char)
    print_wrapped(title)
    print_rule(char)


def print_json_wrapped(obj: object) -> None:
    for line in json.dumps(obj, ensure_ascii=False, indent=2).splitlines():
        leading_spaces = len(line) - len(line.lstrip(" "))
        indent = " " * leading_spaces
        print_wrapped(
            line.strip(),
            initial_indent=indent,
            subsequent_indent=indent + "  ",
            normalize=False,
        )


def pretty_history(messages: list[dict]) -> str:
    if not messages:
        return "NO_HISTORY"
    lines = []
    for message in messages:
        role = (message.get("role") or "unknown").upper()
        content = normalize_whitespace(message.get("content", ""))
        prefix = f"- {role}: "
        lines.append(
            wrap_text(
                content,
                initial_indent=prefix,
                subsequent_indent=" " * len(prefix),
            )
        )
    return "\n".join(lines)


def pretty_state_text(question_text: str, history_prefix: list[dict]) -> str:
    question_block = wrap_text(question_text)
    history_block = pretty_history(history_prefix)
    return f"QUESTION:\n{question_block}\n\nCONVERSATION_SO_FAR:\n{history_block}"


def print_state_text(title: str, state_text: str) -> None:
    print(title)
    for line in state_text.splitlines():
        if not line:
            print()
            continue
        if line.endswith(":") and line.upper() == line:
            print(line)
            continue
        if ": " in line:
            role, content = line.split(": ", 1)
            prefix = f"{role}: "
            print_wrapped(
                content,
                initial_indent=prefix,
                subsequent_indent=" " * len(prefix),
            )
        else:
            print_wrapped(line)


def choose_demo_row(rows: list[dict]) -> dict:
    for row in rows:
        if row.get("conversation_history"):
            return row
    raise RuntimeError("No encontre una fila con conversation_history para la demostracion.")


demo_rows = train_rows if "train_rows" in globals() else load_jsonl(config.data_dir / "train.jsonl")
demo_row = choose_demo_row(demo_rows)
demo_required_points = demo_row.get("required_points") or []
demo_history = demo_row.get("conversation_history") or []
demo_degraded_question = demo_row.get("degraded_question") or demo_row.get("ori_question") or ""

print_section("REGISTRO ORIGINAL SELECCIONADO")
print_json_wrapped(demo_row)

print()
print_section("PASO 1. CAMPOS CLAVE EXTRAIDOS DEL REGISTRO")
print_wrapped(demo_row.get("id", "SIN_ID"), initial_indent="id: ")
print_wrapped(
    demo_row.get("ori_question", ""),
    initial_indent="ori_question: ",
    subsequent_indent=" " * len("ori_question: "),
)
print_wrapped(
    demo_degraded_question,
    initial_indent="degraded_question: ",
    subsequent_indent=" " * len("degraded_question: "),
)
print(f"required_points_total: {len(demo_required_points)}")
print("required_points:")
for idx, point in enumerate(demo_required_points, start=1):
    prefix = f"  {idx}. "
    print_wrapped(point, initial_indent=prefix, subsequent_indent=" " * len(prefix))

print("\nconversation_history original:")
for idx, message in enumerate(demo_history, start=1):
    role = (message.get("role") or "unknown").upper()
    content = normalize_whitespace(message.get("content", ""))
    prefix = f"  {idx}. {role}: "
    print_wrapped(content, initial_indent=prefix, subsequent_indent=" " * len(prefix))

print()
print_section("PASO 2. RECORRIDO DEL HISTORIAL Y CONSTRUCCION DE EJEMPLOS POR TURNO")
assistant_positions = [
    idx for idx, item in enumerate(demo_history) if item.get("role") == "assistant"
 ]
running_history: list[dict] = []
observed_user_evidence: list[str] = []

for index, message in enumerate(demo_history):
    role = message.get("role")
    content = normalize_whitespace(message.get("content", ""))
    print()
    print_rule("-")
    prefix = f"Mensaje original #{index}: role={role}, content="
    print_wrapped(content, initial_indent=prefix, subsequent_indent=" " * len(prefix))

    if role == "user":
        observed_user_evidence.append(message.get("content", ""))
        running_history.append(message)
        evidence_before = normalize_whitespace(" ".join(observed_user_evidence))
        print_wrapped(
            "Accion de preprocesamiento: se acumula como evidencia del usuario "
            "y se agrega al historial corrido."
        )
        print_wrapped(
            evidence_before,
            initial_indent="Evidencia acumulada: ",
            subsequent_indent=" " * len("Evidencia acumulada: "),
        )
        print("Historial corrido disponible:")
        print(pretty_history(running_history))
        continue

    if role != "assistant":
        running_history.append(message)
        print_wrapped(
            "Accion de preprocesamiento: turno ignorado para supervision, "
            "pero se conserva en el historial."
        )
        continue

    assistant_rank = assistant_positions.index(index)
    is_final_assistant = assistant_rank == len(assistant_positions) - 1
    action = "respond" if is_final_assistant else "ask"
    evidence_before = " ".join(observed_user_evidence)
    covered_before = count_covered_points(demo_required_points, evidence_before)
    total_points = max(len(demo_required_points), 1)
    unresolved_fraction = (
        (len(demo_required_points) - covered_before) / total_points
        if demo_required_points
        else 0.0
    )

    print(f"Turno supervisado del asistente #{assistant_rank}")
    print(f"Etiqueta gold: {action}")
    print("Historial visible antes de decidir:")
    print(pretty_history(running_history))
    print(f"covered_before: {covered_before}/{len(demo_required_points)}")
    print(f"unresolved_fraction: {unresolved_fraction:.4f}")

    if action == "ask":
        next_user_content = ""
        for next_message in demo_history[index + 1 :]:
            if next_message.get("role") == "user":
                next_user_content = next_message.get("content", "")
                break
        covered_after = count_covered_points(
            demo_required_points,
            f"{evidence_before} {next_user_content}",
        )
        coverage_gain = max(0, covered_after - covered_before)
        ask_reward = 0.5 + 0.5 * (coverage_gain / total_points)
        respond_reward = -float(unresolved_fraction)
        print_wrapped(
            next_user_content,
            initial_indent="Siguiente respuesta del usuario: ",
            subsequent_indent=" " * len("Siguiente respuesta del usuario: "),
        )
        print(f"covered_after: {covered_after}/{len(demo_required_points)}")
        print(f"coverage_gain: {coverage_gain}")
        print("Explicacion del reward en este turno:")
        print_wrapped(
            "- La etiqueta gold es ask, asi que preguntar parte con reward "
            "positivo base 0.5."
        )
        print_wrapped(
            "- Si la siguiente respuesta del usuario cubre mas required_points, "
            "ask_reward sube segun coverage_gain / total_points."
        )
        print_wrapped(
            "- Responder ya seria prematuro, por eso respond_reward se vuelve "
            "negativo en proporcion a la fraccion no resuelta."
        )
    else:
        ask_reward = -0.5 if demo_required_points else -0.25
        respond_reward = 1.0
        print("Explicacion del reward en este turno:")
        print_wrapped(
            "- La etiqueta gold es respond, asi que responder recibe reward "
            "maximo 1.0."
        )
        print_wrapped(
            "- Seguir preguntando se penaliza porque agregaria costo "
            "conversacional innecesario."
        )
        if demo_required_points:
            print_wrapped(
                "- La penalizacion de ask es -0.5 porque existian "
                "required_points y ya era momento de cerrar."
            )
        else:
            print_wrapped(
                "- La penalizacion de ask es mas suave (-0.25) porque no habia "
                "required_points explicitos."
            )

    state_text = serialize_state_text(demo_degraded_question, running_history)
    print(f"ask_reward: {ask_reward:.4f}")
    print(f"respond_reward: {respond_reward:.4f}")
    print_state_text(
        "State text legible para inspeccion:",
        pretty_state_text(demo_degraded_question, running_history),
    )
    print_state_text(
        "State text exacto generado por serialize_state_text:",
        state_text,
    )

    running_history.append(message)

demo_examples = build_rich_turn_examples(demo_row, split_name="train")
demo_examples_df = pd.DataFrame(demo_examples)
print()
print_section("PASO 3. RESULTADO FINAL DEL PREPROCESAMIENTO")
display(
    demo_examples_df[[
        "source_kind",
        "turn_index",
        "action",
        "covered_before",
        "required_points_total",
        "ask_reward",
        "respond_reward",
        "state_text",
        "gold_response_preview",
    ]]
 )

demo_mlp_rows = demo_examples_df[[
    "example_id",
    "trajectory_id",
    "turn_index",
    "action",
    "label",
    "ask_reward",
    "respond_reward",
    "state_text",
]].copy()
demo_mlp_rows["label_name"] = demo_mlp_rows["label"].map(INDEX_TO_ACTION)
demo_mlp_rows = demo_mlp_rows[[
    "example_id",
    "trajectory_id",
    "turn_index",
    "action",
    "label",
    "label_name",
    "ask_reward",
    "respond_reward",
    "state_text",
]]

print()
print_section("PASO 4. ASI QUEDA ESTE REGISTRO DENTRO DEL DATASET TABULAR QUE CONSUME EL MLP")
print_wrapped("Cada turno del asistente se convierte en una fila del dataset de entrenamiento.")
display(demo_mlp_rows)

if "train_df" in globals() and "validation_df" in globals():
    effective_train_df = train_df
    effective_validation_df = validation_df
else:
    all_examples_for_split = build_training_examples(demo_rows, split_name="train")
    effective_train_df, effective_validation_df = grouped_train_validation_split(
        all_examples_for_split,
        config,
    )

demo_example_ids = set(demo_mlp_rows["example_id"])
demo_train_rows = effective_train_df[
    effective_train_df["example_id"].isin(demo_example_ids)
].copy()
demo_validation_rows = effective_validation_df[
    effective_validation_df["example_id"].isin(demo_example_ids)
].copy()

print()
print_section("PASO 5. UBICACION DE ESTAS FILAS EN EL CONJUNTO REAL USADO POR EL MLP")
print(f"Filas de este registro que quedaron en train: {len(demo_train_rows)}")
print(f"Filas de este registro que quedaron en validation: {len(demo_validation_rows)}")

if not demo_train_rows.empty:
    demo_train_rows["assigned_split"] = "train"
if not demo_validation_rows.empty:
    demo_validation_rows["assigned_split"] = "validation"

assigned_rows = pd.concat([demo_train_rows, demo_validation_rows], ignore_index=True)
if assigned_rows.empty:
    print_wrapped("No se encontraron las filas del ejemplo dentro del split reconstruido.")
else:
    display(
        assigned_rows[[
            "assigned_split",
            "example_id",
            "source_kind",
            "turn_index",
            "action",
            "label",
            "ask_reward",
            "respond_reward",
            "state_text",
        ]]
    )
    print()
    print_wrapped(
        "Estas son exactamente las filas tabulares que luego se vectorizan "
        "y se entregan al MLP."
    )

REGISTRO ORIGINAL SELECCIONADO
{
  "ori_question": "Bernardo randomly picks $3$ distinct numbers from the set
    $\\{1,2,3,4,5,6,7,8,9\\}$ and arranges them in descending order to form a $3$-digit
    number. Silvia randomly picks $3$ distinct numbers from the set
    $\\{1,2,3,4,5,6,7,8\\}$ and also arranges them in descending order to form a $3$-digit
    number. Find the probability that Bernardo's number is larger than Silvia's number.
    The original answer is in \\(\\frac{k}{m}\\) format, please give the value of \\(k +
    m\\).\n\nRemember to put your answer on its own line after \"Answer:\".",
  "expected_answer": "93",
  "pass_rate": 0.0,
  "id": "779cb8a9ce5fc8408050703ebf2068fade636e7354cd2f988904d0d159aa66fa",
  "degraded_question": "Bernardo randomly picks a few distinct numbers from the set
    \\{1,2,3,4,5,6,7,8,9\\} and arranges them in descending order to form a 3-digit
    number. Silvia randomly picks a few distinct numbers from the set
    \\{1,2,3,4,5,6,7,8\\} a

,source_kind,turn_index,action,covered_before,required_points_total,ask_reward,respond_reward,state_text,gold_response_preview
0,degraded_conversation,0,ask,1,3,0.666667,-0.666667,QUESTION:\nBernardo randomly picks a few disti...,How many numbers do Bernardo and Silvia each p...
1,degraded_conversation,1,respond,2,3,-0.500000,1.000000,QUESTION:\nBernardo randomly picks a few disti...,"Bernardo selects 3 distinct numbers from {1,2,..."



PASO 4. ASI QUEDA ESTE REGISTRO DENTRO DEL DATASET TABULAR QUE CONSUME EL MLP
Cada turno del asistente se convierte en una fila del dataset de entrenamiento.


,example_id,trajectory_id,turn_index,action,label,label_name,ask_reward,respond_reward,state_text
0,a54831aae39083bb3112144c09636033487426b1,e0a567a1bd629b8b5a83822427eaf98617c05013,0,ask,0,ask,0.666667,-0.666667,QUESTION:\nBernardo randomly picks a few disti...
1,0ea136a52cdd37d664885d39cbea117239f28c8b,e0a567a1bd629b8b5a83822427eaf98617c05013,1,respond,1,respond,-0.500000,1.000000,QUESTION:\nBernardo randomly picks a few disti...



PASO 5. UBICACION DE ESTAS FILAS EN EL CONJUNTO REAL USADO POR EL MLP
Filas de este registro que quedaron en train: 2
Filas de este registro que quedaron en validation: 0


,assigned_split,example_id,source_kind,turn_index,action,label,ask_reward,respond_reward,state_text
0,train,a54831aae39083bb3112144c09636033487426b1,degraded_conversation,0,ask,0,0.666667,-0.666667,QUESTION:\nBernardo randomly picks a few disti...
1,train,0ea136a52cdd37d664885d39cbea117239f28c8b,degraded_conversation,1,respond,1,-0.500000,1.000000,QUESTION:\nBernardo randomly picks a few disti...



Estas son exactamente las filas tabulares que luego se vectorizan y se entregan al MLP.


## 4. Representacion del estado mediante embeddings

En este notebook no se usa un encoder neuronal preentrenado por defecto. El embedding del estado se construye en dos pasos a partir de `state_text`, que concatena la pregunta degradada y el historial conversacional disponible hasta ese turno.

1. Primero se aplica `TfidfVectorizer` sobre `state_text` usando unigramas y bigramas, con `max_features=4096` y `min_df=2`. Eso convierte cada turno en un vector disperso basado en frecuencia de terminos relevantes del train.
2. Despues se reduce ese vector con `TruncatedSVD` hasta `embedding_dim=256` componentes. Ese vector denso de 256 dimensiones es el que entra al MLP como representacion del estado.

El ajuste (`fit`) del vectorizador y de SVD se hace solo con `train_df` para evitar fuga de informacion hacia validacion. Luego `transform` reutiliza ese encoder para producir `train_features`, `validation_features` y, mas adelante, los features del test oficial.

Si `config.precomputed_embeddings` apunta a un archivo `.npz`, este flujo se reemplaza: el notebook carga directamente los arrays `example_ids` y `embeddings`, y usa esos embeddings externos en lugar del pipeline `TF-IDF + SVD`.

In [9]:
class EmbeddingBuilder:
    def __init__(self, config: BaselineConfig):
        self.config = config
        self.vectorizer: TfidfVectorizer | None = None
        self.svd: TruncatedSVD | None = None
        self.embedding_lookup: dict[str, np.ndarray] | None = None

    def fit(self, train_df: pd.DataFrame) -> None:
        if self.config.precomputed_embeddings is not None:
            payload = np.load(self.config.precomputed_embeddings, allow_pickle=False)
            example_ids = payload["example_ids"]
            embeddings = payload["embeddings"]
            self.embedding_lookup = {
                str(example_id): embedding.astype(np.float32)
                for example_id, embedding in zip(example_ids, embeddings, strict=False)
            }
            return

        self.vectorizer = TfidfVectorizer(
            max_features=self.config.tfidf_max_features,
            ngram_range=(1, 2),
            min_df=2,
        )
        sparse_matrix = self.vectorizer.fit_transform(train_df["state_text"])
        if sparse_matrix.shape[1] <= 1:
            self.svd = None
            return

        max_components = min(
            self.config.embedding_dim,
            max(1, sparse_matrix.shape[0] - 1),
            max(1, sparse_matrix.shape[1] - 1),
        )
        self.svd = TruncatedSVD(n_components=max_components, random_state=self.config.random_seed)
        self.svd.fit(sparse_matrix)

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        if self.embedding_lookup is not None:
            missing = [example_id for example_id in df["example_id"] if example_id not in self.embedding_lookup]
            if missing:
                raise KeyError(
                    "Faltan embeddings precomputados para algunos example_id. "
                    f"Primer faltante: {missing[0]}"
                )
            return np.stack([self.embedding_lookup[example_id] for example_id in df["example_id"]]).astype(np.float32)

        if self.vectorizer is None:
            raise RuntimeError("El encoder textual no fue ajustado todavia.")
        sparse_matrix = self.vectorizer.transform(df["state_text"])
        if self.svd is None:
            return sparse_matrix.toarray().astype(np.float32)
        return self.svd.transform(sparse_matrix).astype(np.float32)

def build_mlp(input_dim: int, output_dim: int, hidden_dims: tuple[int, ...], dropout: float) -> nn.Sequential:
    layers: list[nn.Module] = []
    previous_dim = input_dim
    for hidden_dim in hidden_dims:
        layers.append(nn.Linear(previous_dim, hidden_dim))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        previous_dim = hidden_dim
    layers.append(nn.Linear(previous_dim, output_dim))
    return nn.Sequential(*layers)

class EarlyStopping:
    def __init__(self, patience: int):
        self.patience = patience
        self.best_loss = float("inf")
        self.bad_epochs = 0
        self.best_state: dict[str, torch.Tensor] | None = None

    def update(self, loss: float, model: nn.Module) -> bool:
        if loss < self.best_loss:
            self.best_loss = loss
            self.bad_epochs = 0
            self.best_state = {
                key: value.detach().cpu().clone() for key, value in model.state_dict().items()
            }
            return False
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def restore(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

def make_tensor_dataset(features: np.ndarray, targets: np.ndarray, task: str) -> TensorDataset:
    x_tensor = torch.as_tensor(np.array(features, copy=True), dtype=torch.float32)
    if task == "classification":
        y_tensor = torch.as_tensor(np.array(targets, copy=True), dtype=torch.long)
    else:
        y_tensor = torch.as_tensor(np.array(targets, copy=True), dtype=torch.float32)
    return TensorDataset(x_tensor, y_tensor)

In [10]:
embedding_builder = EmbeddingBuilder(config)
embedding_builder.fit(train_df)

train_features = embedding_builder.transform(train_df)
validation_features = embedding_builder.transform(validation_df)

train_labels = train_df["label"].to_numpy(dtype=np.int64)
validation_labels = validation_df["label"].to_numpy(dtype=np.int64)

train_rewards = train_df[["ask_reward", "respond_reward"]].to_numpy(dtype=np.float32)
validation_rewards = validation_df[["ask_reward", "respond_reward"]].to_numpy(dtype=np.float32)

print("Dimensiones")
print(f"train_features: {train_features.shape}")
print(f"validation_features: {validation_features.shape}")
print(f"device: {config.device}")

Dimensiones
train_features: (9809, 256)
validation_features: (2437, 256)
device: cpu


## 5. Disenio de los baselines

Despues del preprocesamiento, ambos baselines reciben las mismas features y ambos se formulan como aprendizaje por refuerzo sobre acciones discretas `ask/respond`. Como el notebook no tiene un simulador interactivo completo, se usa una version offline/contextual: los estados salen de los turnos observados y los rewards salen de la heuristica con `required_points`.

### Baseline 1: `mlp_policy_gradient`

Este baseline trata el MLP como una politica estocastica:

- entrada: embedding del estado
- salida: logits para una distribucion `pi(ask | s)` y `pi(respond | s)`
- objetivo: maximizar el reward esperado bajo la politica

La loss ya no es `CrossEntropyLoss`. Ahora se calcula la recompensa esperada de la politica sobre los rewards moldeados de cada accion y se agrega una pequena bonificacion de entropia para evitar que la politica colapse demasiado pronto.

### Baseline 2: `mlp_q_learning`

Este baseline trata el MLP como una red de valores de accion:

- entrada: embedding del estado
- salida: `Q(s, ask)` y `Q(s, respond)`
- objetivo: ajustar `Q(s, ask)` con un target temporal `r_ask + gamma * max_a Q(s_next, a)` y `Q(s, respond)` con su reward inmediato

Para construir `s_next`, el preprocesamiento conserva `trajectory_id`, `step_index` e `is_terminal`. Si el turno actual no es terminal, el siguiente estado es el proximo turno del asistente dentro de la misma conversacion; si es terminal, el target usa solo el reward inmediato.

### Como interpretar la comparacion

Los dos modelos optimizan reward, pero desde lecturas distintas:

- `mlp_policy_gradient`: aprende directamente una politica que asigna probabilidad a `ask/respond`
- `mlp_q_learning`: aprende valores de accion y decide con `argmax` sobre `Q(s, a)`

Por eso en los resultados conviene mirar `avg_shaped_reward` como metrica principal, y dejar `accuracy`/`macro_f1` como una referencia de cuanto se parece la politica aprendida a la trayectoria observada.

In [11]:
def build_q_learning_arrays(df: pd.DataFrame, features: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    index_by_step = {
        (row.trajectory_id, int(row.step_index)): index
        for index, row in df.reset_index(drop=True).iterrows()
    }
    next_features = np.zeros_like(features, dtype=np.float32)
    dones = np.ones(len(df), dtype=np.float32)

    for index, row in df.reset_index(drop=True).iterrows():
        if bool(row.is_terminal):
            continue
        next_index = index_by_step.get((row.trajectory_id, int(row.step_index) + 1))
        if next_index is None:
            continue
        next_features[index] = features[next_index]
        dones[index] = 0.0

    reward_matrix = df[["ask_reward", "respond_reward"]].to_numpy(dtype=np.float32)
    return reward_matrix, next_features.astype(np.float32), dones


def train_policy_gradient(
    train_features: np.ndarray,
    train_rewards: np.ndarray,
    validation_features: np.ndarray,
    validation_rewards: np.ndarray,
    config: BaselineConfig,
) -> tuple[nn.Module, pd.DataFrame]:
    device = torch.device(config.device)
    model = build_mlp(train_features.shape[1], len(ACTION_TO_INDEX), config.hidden_dims, config.dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    loader = DataLoader(
        TensorDataset(
            torch.as_tensor(np.array(train_features, copy=True), dtype=torch.float32),
            torch.as_tensor(np.array(train_rewards, copy=True), dtype=torch.float32),
        ),
        batch_size=config.batch_size,
        shuffle=True,
    )
    stopper = EarlyStopping(config.patience)
    history = []

    for epoch in range(config.epochs):
        model.train()
        train_losses = []
        train_expected_rewards = []
        for batch_features, batch_rewards in loader:
            batch_features = batch_features.to(device)
            batch_rewards = batch_rewards.to(device)
            logits = model(batch_features)
            log_probs = torch.log_softmax(logits, dim=1)
            probs = log_probs.exp()
            state_baseline = batch_rewards.mean(dim=1, keepdim=True)
            advantages = batch_rewards - state_baseline
            entropy = -(probs * log_probs).sum(dim=1).mean()
            expected_advantage = (probs * advantages.detach()).sum(dim=1).mean()
            loss = -(expected_advantage + config.policy_entropy_coef * entropy)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))
            train_expected_rewards.append(float((probs.detach() * batch_rewards).sum(dim=1).mean().cpu()))

        model.eval()
        with torch.no_grad():
            validation_reward_tensor = torch.as_tensor(np.array(validation_rewards, copy=True), dtype=torch.float32, device=device)
            validation_logits = model(torch.as_tensor(validation_features, dtype=torch.float32, device=device))
            validation_probs = torch.softmax(validation_logits, dim=1)
            validation_expected_reward = (validation_probs * validation_reward_tensor).sum(dim=1).mean()
            validation_loss = -validation_expected_reward

        validation_loss_value = float(validation_loss.detach().cpu())
        history.append({
            "epoch": epoch + 1,
            "train_loss": float(np.mean(train_losses)),
            "train_expected_reward": float(np.mean(train_expected_rewards)),
            "validation_loss": validation_loss_value,
            "validation_expected_reward": float(validation_expected_reward.detach().cpu()),
        })
        if stopper.update(validation_loss_value, model):
            break

    stopper.restore(model)
    return model, pd.DataFrame(history)


def train_q_learning(
    train_features: np.ndarray,
    train_rewards: np.ndarray,
    train_next_features: np.ndarray,
    train_dones: np.ndarray,
    validation_features: np.ndarray,
    validation_rewards: np.ndarray,
    validation_next_features: np.ndarray,
    validation_dones: np.ndarray,
    config: BaselineConfig,
) -> tuple[nn.Module, pd.DataFrame]:
    device = torch.device(config.device)
    model = build_mlp(train_features.shape[1], len(ACTION_TO_INDEX), config.hidden_dims, config.dropout).to(device)
    target_model = build_mlp(train_features.shape[1], len(ACTION_TO_INDEX), config.hidden_dims, config.dropout).to(device)
    target_model.load_state_dict(model.state_dict())
    target_model.eval()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    criterion = nn.SmoothL1Loss()
    loader = DataLoader(
        TensorDataset(
            torch.as_tensor(np.array(train_features, copy=True), dtype=torch.float32),
            torch.as_tensor(np.array(train_rewards, copy=True), dtype=torch.float32),
            torch.as_tensor(np.array(train_next_features, copy=True), dtype=torch.float32),
            torch.as_tensor(np.array(train_dones, copy=True), dtype=torch.float32),
        ),
        batch_size=config.batch_size,
        shuffle=True,
    )
    stopper = EarlyStopping(config.patience)
    history = []

    for epoch in range(config.epochs):
        model.train()
        train_losses = []
        for batch_features, batch_rewards, batch_next_features, batch_dones in loader:
            batch_features = batch_features.to(device)
            batch_rewards = batch_rewards.to(device)
            batch_next_features = batch_next_features.to(device)
            batch_dones = batch_dones.to(device)
            q_values = model(batch_features)
            with torch.no_grad():
                next_q_values = target_model(batch_next_features).max(dim=1).values
                td_targets = batch_rewards.clone()
                ask_idx = ACTION_TO_INDEX["ask"]
                td_targets[:, ask_idx] = batch_rewards[:, ask_idx] + config.q_discount_factor * (1.0 - batch_dones) * next_q_values
            loss = criterion(q_values, td_targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        target_model.load_state_dict(model.state_dict())
        model.eval()
        with torch.no_grad():
            validation_feature_tensor = torch.as_tensor(validation_features, dtype=torch.float32, device=device)
            validation_reward_tensor = torch.as_tensor(np.array(validation_rewards, copy=True), dtype=torch.float32, device=device)
            validation_next_tensor = torch.as_tensor(validation_next_features, dtype=torch.float32, device=device)
            validation_done_tensor = torch.as_tensor(np.array(validation_dones, copy=True), dtype=torch.float32, device=device)
            validation_q = model(validation_feature_tensor)
            next_validation_q = target_model(validation_next_tensor).max(dim=1).values
            validation_targets = validation_reward_tensor.clone()
            ask_idx = ACTION_TO_INDEX["ask"]
            validation_targets[:, ask_idx] = validation_reward_tensor[:, ask_idx] + config.q_discount_factor * (1.0 - validation_done_tensor) * next_validation_q
            validation_loss = criterion(validation_q, validation_targets)

        validation_loss_value = float(validation_loss.detach().cpu())
        history.append({
            "epoch": epoch + 1,
            "train_loss": float(np.mean(train_losses)),
            "validation_loss": validation_loss_value,
        })
        if stopper.update(validation_loss_value, model):
            break

    stopper.restore(model)
    return model, pd.DataFrame(history)


@torch.no_grad()
def predict_policy(model: nn.Module, features: np.ndarray, device_name: str) -> np.ndarray:
    device = torch.device(device_name)
    model.eval()
    logits = model(torch.as_tensor(features, dtype=torch.float32, device=device))
    return torch.softmax(logits, dim=1).argmax(dim=1).cpu().numpy()


@torch.no_grad()
def predict_q_network(model: nn.Module, features: np.ndarray, device_name: str) -> tuple[np.ndarray, np.ndarray]:
    device = torch.device(device_name)
    model.eval()
    q_values = model(torch.as_tensor(features, dtype=torch.float32, device=device)).cpu().numpy()
    actions = q_values.argmax(axis=1)
    return actions, q_values


def summarize_classification(name: str, labels: np.ndarray, predictions: np.ndarray) -> dict:
    ask_idx = ACTION_TO_INDEX["ask"]
    return {
        "model": name,
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision_macro": float(precision_score(labels, predictions, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(labels, predictions, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "precision_weighted": float(precision_score(labels, predictions, average="weighted", zero_division=0)),
        "recall_weighted": float(recall_score(labels, predictions, average="weighted", zero_division=0)),
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
        "ask_rate": float(np.mean(predictions == ask_idx)),
    }


def summarize_reward(name: str, labels: np.ndarray, predictions: np.ndarray, reward_matrix: np.ndarray) -> dict:
    chosen_rewards = reward_matrix[np.arange(len(predictions)), predictions]
    summary = summarize_classification(name, labels, predictions)
    summary["avg_shaped_reward"] = float(np.mean(chosen_rewards))
    return summary


def build_confusion_table(labels: np.ndarray, predictions: np.ndarray) -> pd.DataFrame:
    action_names = [INDEX_TO_ACTION[index] for index in sorted(INDEX_TO_ACTION)]
    matrix = confusion_matrix(labels, predictions, labels=sorted(INDEX_TO_ACTION))
    return pd.DataFrame(
        matrix,
        index=[f"real_{name}" for name in action_names],
        columns=[f"pred_{name}" for name in action_names],
    )


def display_model_metrics(name: str, labels: np.ndarray, predictions: np.ndarray, reward_matrix: np.ndarray) -> None:
    print(f"Metricas basicas - {name}")
    display(pd.DataFrame([summarize_reward(name, labels, predictions, reward_matrix)]).round(4))
    print("Matriz de confusion")
    display(build_confusion_table(labels, predictions))
    print("Reporte por clase")
    print(
        classification_report(
            labels,
            predictions,
            target_names=[INDEX_TO_ACTION[index] for index in sorted(INDEX_TO_ACTION)],
            digits=3,
            zero_division=0,
        )
    )

In [12]:
train_q_rewards, train_next_features, train_dones = build_q_learning_arrays(train_df, train_features)
validation_q_rewards, validation_next_features, validation_dones = build_q_learning_arrays(validation_df, validation_features)

policy_model, policy_history = train_policy_gradient(
    train_features,
    train_rewards,
    validation_features,
    validation_rewards,
    config,
)
policy_predictions = predict_policy(policy_model, validation_features, config.device)

q_model, q_history = train_q_learning(
    train_features,
    train_q_rewards,
    train_next_features,
    train_dones,
    validation_features,
    validation_q_rewards,
    validation_next_features,
    validation_dones,
    config,
)
q_predictions, q_values = predict_q_network(q_model, validation_features, config.device)

results = pd.DataFrame([
    summarize_reward("mlp_policy_gradient", validation_labels, policy_predictions, validation_rewards),
    summarize_reward("mlp_q_learning", validation_labels, q_predictions, validation_rewards),
]).sort_values(["avg_shaped_reward", "macro_f1", "accuracy"], ascending=False)

metric_columns = [
    "model",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "macro_f1",
    "precision_weighted",
    "recall_weighted",
    "weighted_f1",
    "ask_rate",
    "avg_shaped_reward",
]
print("Resumen comparativo en validacion")
display(results[metric_columns].round(4))

for model_name, predictions in [
    ("mlp_policy_gradient", policy_predictions),
    ("mlp_q_learning", q_predictions),
]:
    display_model_metrics(model_name, validation_labels, predictions, validation_rewards)

best_model_name = results.iloc[0]["model"]
best_predictions = policy_predictions if best_model_name == "mlp_policy_gradient" else q_predictions

print("Historial policy gradient")
display(policy_history.tail(3))
print("Historial Q-learning")
display(q_history.tail(3))

Resumen comparativo en validacion


,model,accuracy,precision_macro,recall_macro,macro_f1,precision_weighted,recall_weighted,weighted_f1,ask_rate,avg_shaped_reward
0,mlp_policy_gradient,0.7645,0.7872,0.7700,0.7619,0.7909,0.7645,0.7609,0.3759,0.5736
1,mlp_q_learning,0.8002,0.7999,0.7995,0.7997,0.8001,0.8002,0.8001,0.5273,0.5468


Metricas basicas - mlp_policy_gradient


,model,accuracy,precision_macro,recall_macro,macro_f1,precision_weighted,recall_weighted,weighted_f1,ask_rate,avg_shaped_reward
0,mlp_policy_gradient,0.7645,0.7872,0.77,0.7619,0.7909,0.7645,0.7609,0.3759,0.5736


Matriz de confusion


...

,pred_ask,pred_respond
real_ask,1033,235
real_respond,252,917


Reporte por clase
              precision    recall  f1-score   support

         ask      0.804     0.815     0.809      1268
     respond      0.796     0.784     0.790      1169

    accuracy                          0.800      2437
   macro avg      0.800     0.800     0.800      2437
weighted avg      0.800     0.800     0.800      2437

Historial policy gradient


,epoch,train_loss,train_expected_reward,validation_loss,validation_expected_reward
2,3,-0.315047,0.551558,-0.570470,0.570470
3,4,-0.330363,0.567443,-0.568815,0.568815
4,5,-0.336809,0.574023,-0.573643,0.573643


Historial Q-learning


,epoch,train_loss,validation_loss
2,3,0.130082,0.126803
3,4,0.121884,0.127854
4,5,0.117462,0.128883


## 6. Resultados de los baselines

Estas graficas resumen el desempeño de ambos baselines en validacion. La primera compara metricas escalares, la segunda muestra donde se equivoca cada modelo con matrices de confusion y la tercera permite revisar si el entrenamiento mejora o se estanca por epoca.


In [13]:
def svg_bar_chart(results_df: pd.DataFrame, columns: list[str]) -> str:
    ordered = results_df.set_index("model").loc[["mlp_policy_gradient", "mlp_q_learning"]]
    colors = {
        "mlp_policy_gradient": "#0f766e",
        "mlp_q_learning": "#b45309",
    }
    model_labels = {
        "mlp_policy_gradient": "Policy gradient",
        "mlp_q_learning": "Q-learning",
    }
    width = 980
    label_width = 190
    plot_width = 560
    row_height = 48
    top = 58
    height = top + row_height * len(columns) + 64
    max_value = max(1.0, float(ordered[columns].to_numpy().max()))
    parts = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">',
        '<rect width="100%" height="100%" fill="#fbfaf7"/>',
        '<text x="24" y="30" font-size="20" font-family="DejaVu Sans, sans-serif" font-weight="700" fill="#1f2933">Comparacion de metricas en validacion</text>',
    ]
    legend_x = label_width + plot_width + 80
    for offset, (model, color) in enumerate(colors.items()):
        y = 22 + offset * 22
        parts.append(f'<rect x="{legend_x}" y="{y - 11}" width="14" height="14" rx="2" fill="{color}"/>')
        parts.append(f'<text x="{legend_x + 22}" y="{y}" font-size="13" font-family="DejaVu Sans, sans-serif" fill="#334155">{model_labels[model]}</text>')

    for index, column in enumerate(columns):
        y = top + index * row_height
        parts.append(f'<text x="24" y="{y + 22}" font-size="13" font-family="DejaVu Sans, sans-serif" fill="#334155">{html.escape(column)}</text>')
        parts.append(f'<line x1="{label_width}" y1="{y + 34}" x2="{label_width + plot_width}" y2="{y + 34}" stroke="#e5e7eb"/>')
        for model_index, model in enumerate(ordered.index):
            value = float(ordered.loc[model, column])
            bar_width = max(2.0, plot_width * value / max_value)
            bar_y = y + 4 + model_index * 18
            parts.append(f'<rect x="{label_width}" y="{bar_y}" width="{bar_width:.2f}" height="14" rx="3" fill="{colors[model]}"/>')
            parts.append(f'<text x="{label_width + bar_width + 8:.2f}" y="{bar_y + 11}" font-size="12" font-family="DejaVu Sans, sans-serif" fill="#334155">{value:.4f}</text>')
    parts.append('</svg>')
    return ''.join(parts)


def svg_confusion_matrices(model_predictions: list[tuple[str, np.ndarray]], labels: np.ndarray) -> str:
    action_names = [INDEX_TO_ACTION[index] for index in sorted(INDEX_TO_ACTION)]
    width = 820
    height = 360
    cell = 82
    gap = 110
    lefts = [120, 120 + 2 * cell + gap]
    top = 116
    parts = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">',
        '<rect width="100%" height="100%" fill="#fbfaf7"/>',
        '<text x="24" y="32" font-size="20" font-family="DejaVu Sans, sans-serif" font-weight="700" fill="#1f2933">Matrices de confusion</text>',
        '<text x="24" y="54" font-size="13" font-family="DejaVu Sans, sans-serif" fill="#64748b">Filas: clase real. Columnas: prediccion del modelo.</text>',
    ]
    for panel_index, (name, predictions) in enumerate(model_predictions):
        matrix = confusion_matrix(labels, predictions, labels=sorted(INDEX_TO_ACTION))
        max_count = max(1, int(matrix.max()))
        left = lefts[panel_index]
        title = 'Policy gradient' if name == 'mlp_policy_gradient' else 'Q-learning'
        title_x = left + cell
        parts.append(f'<text x="{title_x}" y="84" font-size="15" font-family="DejaVu Sans, sans-serif" font-weight="700" text-anchor="middle" fill="#334155">{title}</text>')
        for column_index, action in enumerate(action_names):
            x = left + column_index * cell + cell / 2
            parts.append(f'<text x="{x}" y="{top - 18}" font-size="12" font-family="DejaVu Sans, sans-serif" text-anchor="middle" fill="#475569">pred {action}</text>')
        for row_index, action in enumerate(action_names):
            y = top + row_index * cell + cell / 2
            parts.append(f'<text x="{left - 16}" y="{y + 4}" font-size="12" font-family="DejaVu Sans, sans-serif" text-anchor="end" fill="#475569">real {action}</text>')
            for column_index in range(len(action_names)):
                count = int(matrix[row_index, column_index])
                intensity = count / max_count
                r = int(232 - intensity * 188)
                g = int(245 - intensity * 132)
                b = int(242 - intensity * 126)
                x = left + column_index * cell
                y_cell = top + row_index * cell
                text_color = '#ffffff' if intensity > 0.55 else '#0f172a'
                parts.append(f'<rect x="{x}" y="{y_cell}" width="{cell}" height="{cell}" fill="rgb({r},{g},{b})" stroke="#ffffff" stroke-width="3"/>')
                parts.append(f'<text x="{x + cell / 2}" y="{y_cell + cell / 2 + 5}" font-size="22" font-family="DejaVu Sans, sans-serif" font-weight="700" text-anchor="middle" fill="{text_color}">{count}</text>')
    parts.append('</svg>')
    return ''.join(parts)


def _line_path(values: list[float], x: int, y: int, width: int, height: int, min_value: float, max_value: float) -> str:
    if len(values) == 1:
        px = x + width / 2
        py = y + height / 2
        return f'M {px:.2f} {py:.2f}'
    span = max(max_value - min_value, 1e-9)
    points = []
    for index, value in enumerate(values):
        px = x + width * index / (len(values) - 1)
        py = y + height - ((value - min_value) / span) * height
        points.append(f'{px:.2f},{py:.2f}')
    return 'M ' + ' L '.join(points)


def svg_training_curves(policy_history: pd.DataFrame, q_history: pd.DataFrame) -> str:
    width = 980
    height = 390
    chart_width = 380
    chart_height = 210
    panels = [
        {
            'title': 'Policy gradient: reward esperado',
            'x': 70,
            'y': 82,
            'series': [
                ('train_expected_reward', policy_history['train_expected_reward'].tolist(), '#0f766e'),
                ('validation_expected_reward', policy_history['validation_expected_reward'].tolist(), '#2563eb'),
            ],
        },
        {
            'title': 'Q-learning: loss',
            'x': 560,
            'y': 82,
            'series': [
                ('train_loss', q_history['train_loss'].tolist(), '#b45309'),
                ('validation_loss', q_history['validation_loss'].tolist(), '#7c3aed'),
            ],
        },
    ]
    parts = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">',
        '<rect width="100%" height="100%" fill="#fbfaf7"/>',
        '<text x="24" y="32" font-size="20" font-family="DejaVu Sans, sans-serif" font-weight="700" fill="#1f2933">Curvas de entrenamiento</text>',
    ]
    for panel in panels:
        all_values = [value for _, values, _ in panel['series'] for value in values]
        min_value = min(all_values)
        max_value = max(all_values)
        if min_value == max_value:
            min_value -= 1.0
            max_value += 1.0
        x = panel['x']
        y = panel['y']
        parts.append(f'<text x="{x}" y="{y - 24}" font-size="15" font-family="DejaVu Sans, sans-serif" font-weight="700" fill="#334155">{panel["title"]}</text>')
        parts.append(f'<rect x="{x}" y="{y}" width="{chart_width}" height="{chart_height}" fill="#ffffff" stroke="#e2e8f0"/>')
        for tick in range(5):
            ty = y + chart_height * tick / 4
            value = max_value - (max_value - min_value) * tick / 4
            parts.append(f'<line x1="{x}" y1="{ty:.2f}" x2="{x + chart_width}" y2="{ty:.2f}" stroke="#eef2f7"/>')
            parts.append(f'<text x="{x - 8}" y="{ty + 4:.2f}" font-size="10" font-family="DejaVu Sans, sans-serif" text-anchor="end" fill="#64748b">{value:.2f}</text>')
        for name, values, color in panel['series']:
            path = _line_path(values, x, y, chart_width, chart_height, min_value, max_value)
            parts.append(f'<path d="{path}" fill="none" stroke="{color}" stroke-width="3"/>')
            for index, value in enumerate(values):
                px = x + chart_width * index / max(len(values) - 1, 1)
                py = y + chart_height - ((value - min_value) / max(max_value - min_value, 1e-9)) * chart_height
                parts.append(f'<circle cx="{px:.2f}" cy="{py:.2f}" r="4" fill="{color}"/>')
        legend_y = y + chart_height + 34
        for index, (name, _, color) in enumerate(panel['series']):
            legend_label = 'Train' if name.startswith('train_') else 'Validation'
            lx = x + index * 140
            parts.append(f'<rect x="{lx}" y="{legend_y - 11}" width="14" height="14" rx="2" fill="{color}"/>')
            parts.append(f'<text x="{lx + 20}" y="{legend_y}" font-size="12" font-family="DejaVu Sans, sans-serif" fill="#475569">{legend_label}</text>')
    parts.append('</svg>')
    return ''.join(parts)


performance_metric_columns = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "macro_f1",
    "weighted_f1",
    "ask_rate",
    "avg_shaped_reward",
]

model_predictions = [
    ("mlp_policy_gradient", policy_predictions),
    ("mlp_q_learning", q_predictions),
]

display(HTML(svg_bar_chart(results, performance_metric_columns)))
display(HTML(svg_confusion_matrices(model_predictions, validation_labels)))
display(HTML(svg_training_curves(policy_history, q_history)))


In [14]:
official_test_df = build_official_test_examples(official_test_rows)
official_test_features = embedding_builder.transform(official_test_df)

official_test_predictions = (
    predict_policy(policy_model, official_test_features, config.device)
    if best_model_name == "mlp_policy_gradient"
    else predict_q_network(q_model, official_test_features, config.device)[0]
)

ask_rate_test = float(np.mean(official_test_predictions == ACTION_TO_INDEX["ask"]))
test_prediction_summary = (
    pd.Series([INDEX_TO_ACTION[index] for index in official_test_predictions])
    .value_counts()
    .rename_axis("predicted_action")
    .reset_index(name="count")
)

print(f"ask_rate_test_oficial: {ask_rate_test:.3f}")
display(test_prediction_summary)

history_snapshot = {
    "policy_history": policy_history.tail(3).to_dict(orient="records"),
    "q_history": q_history.tail(3).to_dict(orient="records"),
}
history_snapshot

ask_rate_test_oficial: 0.043


,predicted_action,count
0,respond,382
1,ask,17


{'policy_history': [{'epoch': 3,
   'train_loss': -0.3150471944313545,
   'train_expected_reward': 0.5515580556609414,
   'validation_loss': -0.5704699158668518,
   'validation_expected_reward': 0.5704699158668518},
  {'epoch': 4,
   'train_loss': -0.33036292315303506,
   'train_expected_reward': 0.5674432922493328,
   'validation_loss': -0.5688152313232422,
   'validation_expected_reward': 0.5688152313232422},
  {'epoch': 5,
   'train_loss': -0.33680854447476277,
   'train_expected_reward': 0.5740233210774212,
   'validation_loss': -0.5736433863639832,
   'validation_expected_reward': 0.5736433863639832}],
 'q_history': [{'epoch': 3,
   'train_loss': 0.13008239439555577,
   'validation_loss': 0.1268034726381302},
  {'epoch': 4,
   'train_loss': 0.12188380811508599,
   'validation_loss': 0.127854123711586},
  {'epoch': 5,
   'train_loss': 0.11746173235890153,
   'validation_loss': 0.12888330221176147}]}

## 7. Ablaciones

### 7.1 Con historial conversacional vs sin historial conversacional

Esta ablación compara el modelo con el estado completo (`QUESTION + CONVERSATION_SO_FAR`) contra un estado reducido que solo ve la pregunta degradada (`QUESTION` sin historial). El objetivo es medir cuánto contribuye el historial acumulado a la decisión `ask/respond`.

In [15]:
# ── Ablación 1: Con historial vs sin historial ──────────────────────────
# Se construye un estado alternativo que ignora el historial conversacional.
# El resto del pipeline (TF-IDF + SVD + MLP supervisado) es idéntico.

def build_question_only_features(
    df: pd.DataFrame,
    vectorizer: TfidfVectorizer,
    svd: TruncatedSVD | None,
) -> np.ndarray:
    """Vectoriza solo la pregunta degradada, ignorando el historial."""
    question_only_texts = df["state_text"].apply(
        lambda text: text.split("\n\nCONVERSATION_SO_FAR:")[0].strip()
    )
    sparse = vectorizer.transform(question_only_texts)
    if svd is None:
        return sparse.toarray().astype(np.float32)
    return svd.transform(sparse).astype(np.float32)

# Reutiliza el vectorizador ya ajustado sobre train_df
train_features_no_hist = build_question_only_features(
    train_df, embedding_builder.vectorizer, embedding_builder.svd
)
val_features_no_hist = build_question_only_features(
    validation_df, embedding_builder.vectorizer, embedding_builder.svd
)

# Entrena el modelo de policy gradient con features sin historial
set_seed(config.random_seed)
pg_no_hist, _ = train_policy_gradient(
    train_features_no_hist,
    train_rewards,
    val_features_no_hist,
    validation_rewards,
    config,
)
preds_no_hist = predict_policy(pg_no_hist, val_features_no_hist, config.device)

# Entrena el modelo de policy gradient con features completos (historial incluido)
set_seed(config.random_seed)
pg_with_hist, _ = train_policy_gradient(
    train_features,
    train_rewards,
    validation_features,
    validation_rewards,
    config,
)
preds_with_hist = predict_policy(pg_with_hist, validation_features, config.device)

ablation1_results = pd.DataFrame([
    summarize_reward("Sin historial (solo question)", validation_labels, preds_no_hist, validation_rewards),
    summarize_reward("Con historial (question + conv)", validation_labels, preds_with_hist, validation_rewards),
]).round(4)

ablation_cols = ["model", "accuracy", "macro_f1", "ask_rate", "avg_shaped_reward"]
print("Ablación 1 — impacto del historial conversacional")
display(ablation1_results[ablation_cols])

Ablación 1 — impacto del historial conversacional


,model,accuracy,macro_f1,ask_rate,avg_shaped_reward
0,Sin historial (solo question),0.4797,0.3242,0.0000,0.3805
1,Con historial (question + conv),0.7645,0.7619,0.3759,0.5736


Podemos ver la importancia de tener el historial incorporado, pues nuestra avg_shaped_reward es más del doble cuando lo consideramos que cuando no, lo cual hace sentido, pues justo el historial es lo que nos permite ganar la información necesaria a lo largo de las preguntas.

### 7.2 Efecto del costo de preguntar sobre `ask_rate` y reward

Esta ablación varía el costo adicional de preguntar aplicando un descuento sobre `ask_reward`. Con costo bajo el modelo tiene libertad de preguntar; con costo alto se desalienta. Se entrena un MLP de policy gradient bajo cada régimen y se compara el `ask_rate` y el reward promedio.

In [16]:
# ── Ablación 2: Costo de preguntar bajo / medio / alto ──────────────────
# Se reescala ask_reward multiplicando por un factor: 1.0 (bajo), 0.5 (medio), 0.0 (alto).
# respond_reward no se modifica.

def apply_ask_cost(rewards: np.ndarray, ask_multiplier: float) -> np.ndarray:
    """Escala ask_reward por ask_multiplier para simular distintos costos de preguntar."""
    modified = rewards.copy()
    modified[:, ACTION_TO_INDEX["ask"]] *= ask_multiplier
    return modified

ask_cost_configs = [
    ("Costo bajo  (x1.0)",  1.0),
    ("Costo medio (x0.5)",  0.5),
    ("Costo alto  (x0.0)",  0.0),
]

ablation2_rows = []
for label, multiplier in ask_cost_configs:
    set_seed(config.random_seed)
    train_r_scaled = apply_ask_cost(train_rewards, multiplier)
    val_r_scaled   = apply_ask_cost(validation_rewards, multiplier)
    model_cost, _ = train_policy_gradient(
        train_features, train_r_scaled,
        validation_features, val_r_scaled,
        config,
    )
    preds_cost = predict_policy(model_cost, validation_features, config.device)
    row = summarize_reward(label, validation_labels, preds_cost, val_r_scaled)
    ablation2_rows.append(row)

ablation2_results = pd.DataFrame(ablation2_rows).round(4)
ablation_cols2 = ["model", "accuracy", "macro_f1", "ask_rate", "avg_shaped_reward"]
print("Ablación 2 — efecto del costo de preguntar sobre ask_rate y reward")
display(ablation2_results[ablation_cols2])

Ablación 2 — efecto del costo de preguntar sobre ask_rate y reward


,model,accuracy,macro_f1,ask_rate,avg_shaped_reward
0,Costo bajo (x1.0),0.7645,0.7619,0.3759,0.5736
1,Costo medio (x0.5),0.7538,0.7489,0.3398,0.5020
2,Costo alto (x0.0),0.6795,0.6550,0.2130,0.4362


Vemos que incrementar el costo resulta en una perdida de desempeño del modelo, lo que apunta a que justo el balance adecuado de las recompensas para tener un modelo efectivo debe estar construído alrededor de la idea de que preguntar no debe ser tan penalizado como responder incorrectamente.

## 8. Error analysis

Las siguientes celdas imprimen ejemplos del conjunto de validación donde el modelo (`mlp_policy_gradient`) se equivoca, para facilitar el análisis manual. Se muestran por separado:

- **Falsos negativos de ASK**: el modelo predijo `respond` pero la acción correcta era `ask` (respondió prematuramente).
- **Falsos positivos de ASK**: el modelo predijo `ask` pero la acción correcta era `respond` (preguntó de más).

Para cada caso se imprime la pregunta degradada, el historial acumulado, los `required_points`, la acción real y la predicción.

In [20]:
# ── Error analysis — ejemplos de errores del mlp_policy_gradient ────────
# Se usan las predicciones del mejor modelo en validación.

N_EXAMPLES = 5  # número de ejemplos a mostrar por tipo de error

error_df = validation_df.copy()
error_df["prediction"] = preds_with_hist
error_df["pred_action"] = error_df["prediction"].map(INDEX_TO_ACTION)

# Falsos negativos de ASK: label=ask, pred=respond → respondió antes de aclarar
fn_ask = error_df[
    (error_df["action"] == "ask") & (error_df["pred_action"] == "respond")
].reset_index(drop=True)

# Falsos positivos de ASK: label=respond, pred=ask → preguntó de más
fp_ask = error_df[
    (error_df["action"] == "respond") & (error_df["pred_action"] == "ask")
].reset_index(drop=True)

def print_error_case(idx: int, row: pd.Series, error_type: str) -> None:
    print(f"{'─'*80}")
    print(f"Caso {idx+1} | Tipo: {error_type}")
    # Extraer question y conversation del state_text serializado
    state = row["state_text"]
    if "\n\nCONVERSATION_SO_FAR:" in state:
        q_part, conv_part = state.split("\n\nCONVERSATION_SO_FAR:", 1)
        question_text = q_part.replace("QUESTION:\n", "").strip()
        conv_text = conv_part.strip()
    else:
        question_text = state.replace("QUESTION:\n", "").strip()
        conv_text = "NO_HISTORY"
    print(f"  Pregunta degradada : {question_text[:200]}")
    print(f"  Historial acumulado: {conv_text[:2000]}")
    print(f"  required_points    : {row.get('required_points_total', '?')} punto(s)")
    print(f"  Acción correcta    : {row['action']}")
    print(f"  Predicción modelo  : {row['pred_action']}")
    if error_type == "FN_ASK":
        print("  → Error: respondió antes de aclarar")
    else:
        print("  → Error: preguntó de más")

print(f"Total errores FN_ASK (respondió prematuramente): {len(fn_ask)}")
print(f"Total errores FP_ASK (preguntó de más):          {len(fp_ask)}")
print()

print("=" * 80)
print("TIPO A — Respondió antes de aclarar (label=ask, pred=respond)")
print("=" * 80)
for i, row in fn_ask.head(N_EXAMPLES).iterrows():
    print_error_case(i, row, "FN_ASK")
print()

print("=" * 80)
print("TIPO B — Preguntó de más (label=respond, pred=ask)")
print("=" * 80)
for i, row in fp_ask.head(N_EXAMPLES).iterrows():
    print_error_case(i, row, "FP_ASK")

Total errores FN_ASK (respondió prematuramente): 463
Total errores FP_ASK (preguntó de más):          111

TIPO A — Respondió antes de aclarar (label=ask, pred=respond)
────────────────────────────────────────────────────────────────────────────────
Caso 1 | Tipo: FN_ASK
  Pregunta degradada : Define $L_n$ as the least common multiple of all the integers from $1$ to $n$ inclusive. There is a unique integer $h$ such that $$\frac{1}{1}+\frac{1}{2}+\frac{1}{3}+\cdots+\frac{1}{n}=\frac{h}{L_{n}
  Historial acumulado: USER: Define $L_n$ as the least common multiple of all the integers from $1$ to $n$ inclusive. There is a unique integer $h$ such that $$\frac{1}{1}+\frac{1}{2}+\frac{1}{3}+\cdots+\frac{1}{n}=\frac{h}{L_{n}}$$ What is the remainder when $h$ is divided by $17$? Remember to put your answer on its own line after "Answer:".
  required_points    : 3 punto(s)
  Acción correcta    : ask
  Predicción modelo  : respond
  → Error: respondió antes de aclarar
─────────────────────────────

================================================================================
TIPO A — Respondió antes de aclarar (label=ask, pred=respond)
================================================================================
────────────────────────────────────────────────────────────────────────────────
Caso 1 | Tipo: FN_ASK
  Pregunta degradada : Define $L_n$ as the least common multiple of all the integers from $1$ to $n$ inclusive. There is a unique integer $h$ such that
  $$\frac{1}{1}+\frac{1}{2}+\frac{1}{3}+\cdots+\frac{1}{n}=\frac{h}{L_{n}}$$
  Historial acumulado: USER: Define $L_n$ as the least common multiple of all the integers from $1$ to $n$ inclusive. There is a unique integer $h$ such that $$\frac{1}{1}+\frac{1}{2}+\frac{1}{3}+\cdots+\frac{1}{n}=\frac{h}{L_{n}}$$ What is the remainder when $h$ is divided by $17$? Remember to put your answer on its own line after "Answer:".
  `required_points`    : 3 punto(s)
  Acción correcta    : ask
  Predicción modelo  : respond
  → Error: respondió antes de aclarar
────────────────────────────────────────────────────────────────────────────────
Caso 2 | Tipo: FN_ASK
  Pregunta degradada : A deck of forty cards consists of four $1$'s, four $2$'s,..., and four $10$'s. A couple of cards with the same number is removed from the deck. Given that these cards are not returned to the deck, let
  Historial acumulado: USER: A deck of forty cards consists of four $1$'s, four $2$'s,..., and four $10$'s. A couple of cards with the same number is removed from the deck. Given that these cards are not returned to the deck, let $m/n$ be the probability that two randomly selected cards also form a pair, where $m$ and $n$ are relatively prime positive integers. Find $m + n.$
  `required_points`    : 4 punto(s)
  Acción correcta    : ask
  Predicción modelo  : respond
  → Error: respondió antes de aclarar
────────────────────────────────────────────────────────────────────────────────
Caso 3 | Tipo: FN_ASK
  Pregunta degradada : In convex hexagon $ABCDEF$, all sides are the same length, $\angle A$ and $\angle D$ are 90-degree angles, and $\angle B$, $\angle C$, $\angle E$, and $\angle F$ are congruent. The area of the hexagon
  Historial acumulado: USER: In convex hexagon $ABCDEF$, all sides are the same length, $\angle A$ and $\angle D$ are 90-degree angles, and $\angle B$, $\angle C$, $\angle E$, and $\angle F$ are congruent. The area of the hexagonal region is some number times ($\sqrt{2}+1$). Find $AB$.
  `required_points`    : 3 punto(s)
  Acción correcta    : ask
  Predicción modelo  : respond
  → Error: respondió antes de aclarar
────────────────────────────────────────────────────────────────────────────────
Caso 4 | Tipo: FN_ASK
  Pregunta degradada : A finite set $M$ of positive integers consists of distinct perfect squares and the number $92$. The average of the numbers in $M$ is a bit over 80. If we remove $92$ from $M$, the average drops to a s
  Historial acumulado: USER: A finite set $M$ of positive integers consists of distinct perfect squares and the number $92$. The average of the numbers in $M$ is a bit over 80. If we remove $92$ from $M$, the average drops to a slightly lower value. If $N^2$ is the largest possible square in $M$, what is the value of $N$?
  `required_points`    : 3 punto(s)
  Acción correcta    : ask
  Predicción modelo  : respond
  → Error: respondió antes de aclarar
────────────────────────────────────────────────────────────────────────────────
Caso 5 | Tipo: FN_ASK
  Pregunta degradada : A hotel packed breakfast for each of a few guests. Each breakfast should have consisted of three types of rolls, one of each kind. The preparer wrapped each of the rolls and once wrapped, the rolls we
  Historial acumulado: USER: A hotel packed breakfast for each of a few guests. Each breakfast should have consisted of three types of rolls, one of each kind. The preparer wrapped each of the rolls and once wrapped, the rolls were indistinguishable from one another. She then randomly put three rolls in a bag for each of the guests. Given that the probability each guest got one roll of each type is $\frac mn,$ where $m$ and $n$ are relatively prime integers, find $m+n.$ Remember to put your answer on its own line after "Answer:".
  `required_points`    : 4 punto(s)
  Acción correcta    : ask
  Predicción modelo  : respond
  → Error: respondió antes de aclarar

================================================================================
TIPO B — Preguntó de más (label=respond, pred=ask)
================================================================================
────────────────────────────────────────────────────────────────────────────────
Caso 1 | Tipo: FP_ASK
  Pregunta degradada : Evaluate $\frac{\log_{\text{some number}}\text{next number}^2 \cdot \log_{\text{next number}}\text{following number}^2 \cdot \log_{\text{following number}}\text{next one}^2 \cdot \cdot \cdot \log_{\text{before last}}\text{last number}^2}{\log_{\text{some start}}\text{next value}^2 \cdot \log_{\text{next value}}\text{next next}^2 \cdot \log_{\text{next next}}\text{next one}^2 \cdot \cdot \cdot \log_{\text{second to last}}\text{last term}^2}$ . Remember to put your answer on its own line after "Answer:".
  Historial acumulado: USER: Evaluate $\frac{\log_{\text{some number}}\text{next number}^2 \cdot \log_{\text{next number}}\text{following number}^2 \cdot \log_{\text{following number}}\text{next one}^2 \cdot \cdot \cdot \log_{\text{before last}}\text{last number}^2}{\log_{\text{some start}}\text{next value}^2 \cdot \log_{\text{next value}}\text{next next}^2 \cdot \log_{\text{next next}}\text{next one}^2 \cdot \cdot \cdot \log_{\text{second to last}}\text{last term}^2}$ . Remember to put your answer on its own line after "Answer:".
ASSISTANT: Answer: Could you please clarify the exact sequence of numbers used in the bases and arguments of the logarithms in both the numerator and denominator, including the starting values, step sizes, and ending points?
USER: The numerator involves logarithms with bases and arguments that are consecutive multiples of 10, starting from 10 and going up to 1000 (i.e., $\log_{10}20^2 \cdot \log_{20}30^2 \cdot \dots \cdot \log_{990}1000^2$). The denominator involves logarithms with bases and arguments that are consecutive integers, starting from 10 and going up to 100 (i.e., $\log_{10}11^2 \cdot \log_{11}12^2 \cdot \dots \cdot \log_{99}100^2$).
  `required_points`    : 5 punto(s)
  Acción correcta    : respond
  Predicción modelo  : ask
  → Error: preguntó de más
────────────────────────────────────────────────────────────────────────────────
Caso 2 | Tipo: FP_ASK
  Pregunta degradada : Suppose that $x_1 + 1 = x_2 + 2 = x_3 + 3 = \cdots = x_{\text{a large number}} + \text{a large number} = x_1 + x_2 + x_3 + \cdots + x_{\text{a large number}} + \text{one more than that}$. Find the value of $\left\lfloor |S| \right\rfloor$, where $S = \displaystyle\sum_{n=1}^{\text{a large number}} x_n$. Remember to put
  Historial acumulado: USER: Suppose that $x_1 + 1 = x_2 + 2 = x_3 + 3 = \cdots = x_{\text{a large number}} + \text{a large number} = x_1 + x_2 + x_3 + \cdots + x_{\text{a large number}} + \text{one more than that}$. Find the value of $\left\lfloor |S| \right\rfloor$, where $S = \displaystyle\sum_{n=1}^{\text{a large number}} x_n$. Remember to put your answer on its own line after "Answer:".
ASSISTANT: What is the actual number of terms in the sequence, and what does "one more than that" refer to in the final equality?
USER: The number of terms is 2008, and "one more than that" refers to 2009.
  `required_points`    : 3 punto(s)
  Acción correcta    : respond
  Predicción modelo  : ask
  → Error: preguntó de más
────────────────────────────────────────────────────────────────────────────────
Caso 3 | Tipo: FP_ASK
  Pregunta degradada : What is the greatest number of consecutive integers whose sum is a small sum? Remember to put your answer on its own line after "Answer:".
  Historial acumulado: USER: What is the greatest number of consecutive integers whose sum is a small sum? Remember to put your answer on its own line after "Answer:".
ASSISTANT: Answer: Could you please clarify what you mean by "a small sum"—specifically, do you have a particular numerical value in mind for the sum of the consecutive integers?
USER: Answer: The sum I'm referring to is 45.
  `required_points`    : 2 punto(s)
  Acción correcta    : respond
  Predicción modelo  : ask
  → Error: preguntó de más
────────────────────────────────────────────────────────────────────────────────
Caso 4 | Tipo: FP_ASK
  Pregunta degradada : Let $a$ and $b$ be positive integers satisfying $a^2 + ab + b^2 = \text{something like } (b + \text{a small number})^2 + \text{a bit more}.$ Find the minimum possible value of $a + b.$ Remember to put
  Historial acumulado: USER: Let $a$ and $b$ be positive integers satisfying $a^2 + ab + b^2 = \text{something like } (b + \text{a small number})^2 + \text{a bit more}.$ Find the minimum possible value of $a + b.$ Remember to put your answer on its own line after "Answer:".
ASSISTANT: What is the precise form of the expression on the right-hand side of the equation? Specifically, could you clarify what "something like $(b + \text{a small number})^2 + \text{a bit more}$" means in exact terms?
USER: The expression is exactly $(b + 3)^2 + 27$, and there's also the condition that $3a < b$.
  `required_points`    : 3 punto(s)
  Acción correcta    : respond
  Predicción modelo  : ask
  → Error: preguntó de más
────────────────────────────────────────────────────────────────────────────────
Caso 5 | Tipo: FP_ASK
  Pregunta degradada : The product of two positive palindromes with several digits is . What is their sum? Remember to put your answer on its own line after "Answer:".
  Historial acumulado: USER: The product of two positive palindromes with several digits is . What is their sum? Remember to put your answer on its own line after "Answer:".
ASSISTANT: Answer: What is the exact product of the two palindromes, and how many digits does each palindrome have?
USER: The product of the two palindromes is 436,995, and both are three-digit palindromes.
  `required_points`    : 3 punto(s)
  Acción correcta    : respond
  Predicción modelo  : ask
  → Error: preguntó de más

**Tipo A — Respuesta Prematura (label=ask, pred=respond)**


El modelo tiende a responder directamente cuando la descripción de las variables o el enunciado del problema es suficientemente completo en apariencia. Dado que el historial ya contiene una versión relativamente bien formada del problema, el modelo no encuentra señal clara para seguir preguntando y toma la decisión de responder antes de haber cubierto los puntos requeridos. Si el modelo fuera capaz de mantener una representación más interna de las variables a discernir durante el problema (con estrategias de razonamiento como CoT) podría ser capaz de identificar mejor la falta de información necesaria para responder. También el diseño del Prompt "Remember to put your answer on its own line after "Answer:"." podría estar incitando al modelo a responder antes de tiempo, y en este caso podría ser útil tomar una indicación más neutra que incite al modelo tanto a responder como a preguntar. En el contexto de RL, podríamos tratar de resolver este problema integrando un diseño de recompensas más elaborado, donde hagamos un rebalance hacia obtener respuestas correctas, dándole menos penalización a preguntar, pues es preferible tener una respuesta correcta en más tiempo que tener una respuesta incorrecta rápida.


**Tipo B — Pregunta Innecesaria (label=respond, pred=ask)**

El modelo tiende a continuar preguntando cuando la descripción de las variables en el historial es "verbosa" o extensa. De nuevo, si el modelo guardara un estado interno de la representación de las variables que importan para el problema, podría identificar más fácil cuando ya tuviera las variables necesarias. Parece que se abstiene a responder inmediatamente cuando las respuestas no son simples problemas algebráicos sino que requieren una cierta exploración matemática del problema bajo las condiciones impuestas, esto, en el contexto de RL, podría tratarse de solucionar asignando una nueva acción de _pensar_ (con CoT, por ejemplo) con sus propias recompensas positivas en el caso de que sea necesario, para enseñar al modelo a incentivar la exploración de las variables del problema antes de decidir si responder o preguntar.